# Week 4: Attention

Attention may be the most important idea in transformers, and it's a [much broader idea](https://en.wikipedia.org/wiki/Attention_(machine_learning)) beyond its use in transformers alone.

Inside a transformer, attention is the one step where positions communicate. Everything else (the embedding lookup, the feed-forward layers) processes each position independently. Attention is what lets one position pull in information from the others positions, so a word's embedding can become aware of its context.

Today we'll understand attention as its own thing first, then plug it into the bigger picture of tokens and embeddings next week.

So what is attention?  
**It's a "*soft* dictionary."**

First, recall how a Python dictionary works:

```python
my_dict = {'a': 1, 'b': 2, 'c': 3}

print(my_dict['b'])  # 2
```

You store **keys** (`a`, `b`, `c`), match them against a **query** (`'b'`), and get back a **value** (`2`).

But this only works on exact matches:

```python
my_dict['bb']  # KeyError
```

Embeddings don't work that way. `cat` and `kitten` are *close* but never *equal*, so what would it mean to look something up by "approximately this"? **And how could we train such a lookup?**

Let's build attention from scratch to find out!

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

### Recreating Python Dictionaries Via Orthogonal Vectors, Dot Products, and Softmax

In [ ]:
def one_hot(i):
    return np.eye(3)[i]

mappings = [
    (one_hot(0), one_hot(1)),
    (one_hot(1), one_hot(2)),
    (one_hot(2), one_hot(0))
]

query = one_hot(1)

# score the query against all keys
scores = [np.dot(query, k) for k, v in mappings]
print(scores)

# get sum of all values weighted by the scores
retrieval = sum(dot * v for dot, (k, v) in zip(scores, mappings))
print(retrieval)

In [ ]:
# The same can be done with pure linear algebra

mappings = [
    (one_hot(0), one_hot(1)),
    (one_hot(1), one_hot(2)),
    (one_hot(2), one_hot(0))
]

query = one_hot(1)

keys = np.array([k for k, v in mappings])
values = np.array([v for k, v in mappings])
scores = keys @ query
retrieval = scores @ values
print(retrieval)

### Using nearly-orthogonal vectors

Recall the "curse of dimensionality" from last week. In high-dimensional spaces, random vectors tend to be nearly orthogonal to each other.

We can use this fact to store exponentially more mappings than there are dimensions in the space.

In [ ]:
def generate_nearly_orthogonal_vectors(dim, num_vectors):
    # generate more than needed and keep selecting the next-best nearly orthogonal vector until we have them all
    # this is a simple way to generate nearly orthogonal vectors by greedily selecting the next-best candidate
    pool = np.random.randn(num_vectors * 10, dim)
    pool /= np.linalg.norm(pool, axis=1, keepdims=True)
    vectors = []
    highest_dot = 0
    while len(vectors) < num_vectors:
        best_candidate = None
        best_dot = 1000
        for candidate in pool:
            dot = max(abs(np.dot(candidate, v)) for v in vectors) if vectors else 0
            if dot < best_dot:
                best_dot = dot
                best_candidate = candidate
        vectors.append(best_candidate)
        highest_dot = max(highest_dot, best_dot)
    print("Highest dot product among selected vectors:", highest_dot)
    return np.array(vectors)

vectors = generate_nearly_orthogonal_vectors(dim=32, num_vectors=10)

mappings = [
    (vectors[0], vectors[1]),
    (vectors[1], vectors[2]),
    (vectors[2], vectors[3]),
    (vectors[3], vectors[4]),
    (vectors[4], vectors[5]),
    (vectors[5], vectors[6]),
    (vectors[6], vectors[7]),
    (vectors[7], vectors[8]),
    (vectors[8], vectors[9]),
    (vectors[9], vectors[0])
]

In [ ]:
def attention(mappings, query):
    keys = np.array([k for k, v in mappings])
    values = np.array([v for k, v in mappings])
    scores = keys @ query
    retrieval = scores @ values
    return scores, retrieval

scores, retrieval = attention(mappings, vectors[4])

# What do the raw results look like?
print("Retrieval:", retrieval)

# rather than looking at the retrieval itself (which is just random), we can look at how it **in terms of** all the values.
# The retrieval is like a set of "what vectors are present", and we want the single intended mapped value to be the most present.
plt.bar(range(len(scores)), scores)
plt.show()
for i, value in enumerate(v for k, v in mappings):
    presence = retrieval @ value
    print(f"Presence of value {i} in retrieval: {presence:.2f}")
print('-'*50)

### Softmax attention

We should again consider what dot products actually look like as scores. They are the cosine of the angle between vectors (since the vectors are normalized).

This means they can span between -1 and 1.  
Downsides of this: The differences between scores can be small, and values can actively cancel each other out in the retrieval.

Imagine you have keys embedding opposite words, meaning their scores are opposite (like 1 and -1):
- `hot: abc`
- `cold: abc`

What would a regular dictionary return for a query `hot`?  
The answer is `abc`.

But our current attention implementation would not return that. It would return:
- `dot(hot,hot)*abc + dot(cold,hot)*abc`
- `1*abc + -1*abc`
- `abc - abc`
- `0`

What if we change it to a probability distribution using [softmax](https://www.desmos.com/calculator/pmko7malg0)?

Two advantages:
1. softmax can more strongly emphasize just the highest scoring keys, making the retrieval more focused (less noise!)
2. **the lowest scores go to zero and never go negative.** This means values can never subtract from the result. A negative score becomes "no effect" rather than "actively remove"

Overall, softmax attention is exactly the same as regular (linear) attention, just with softmax applied to the scores (dot products).

Another intuitive way to think about it: softmax turns dot products into a *probability distribution*. So it's almost like we're asking for each key's *match probability* within the set of all keys. And rather than choosing one to be the match, the result is a *blend* of all values weighted by how much their keys matched the query.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1280 650" width="100%" style="max-width:1280px" font-family="system-ui, -apple-system, Segoe UI, Helvetica, Arial, sans-serif" role="img" aria-label="Diagram: a Python dictionary does exact-match lookup, while attention is a soft dictionary that blends values by similarity">
<text x="50" y="32" font-size="20" text-anchor="start" font-weight="600" fill="currentColor">From a dictionary to softmax attention</text>
<!-- ============ PANEL 1: Python dict ============ -->
<rect x="50" y="60" width="1010" height="200" rx="14" fill="currentColor" fill-opacity="0.025" stroke="currentColor" stroke-opacity="0.35" stroke-width="1.5"/>
<text x="70" y="84" font-size="14" text-anchor="start" font-weight="600" fill="currentColor">Python dict: exact match</text>
<text x="115" y="102" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">query</text>
<text x="340" y="102" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">keys</text>
<text x="480" y="102" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">match?</text>
<text x="640" y="102" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">values</text>
<text x="840" y="102" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">result</text>
<!-- query -->
<rect x="70" y="154" width="90" height="32" rx="6" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.5"/>
<text x="115" y="175" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">'b'</text>
<!-- query -> keys -->
<path d="M160,170 H230 V125 H293" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-opacity="0.3" stroke-linejoin="round"/>
<polygon points="299,125 292,121.5 292,128.5" fill="#E8833A" fill-opacity="0.3"/>
<path d="M160,170 H230 V215 H293" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-opacity="0.3" stroke-linejoin="round"/>
<polygon points="299,215 292,211.5 292,218.5" fill="#E8833A" fill-opacity="0.3"/>
<path d="M160,170 H293" fill="none" stroke="#E8833A" stroke-width="2.2" stroke-linejoin="round"/>
<polygon points="299,170 292,166.5 292,173.5" fill="#E8833A"/>
<!-- keys -->
<rect x="300" y="109" width="80" height="32" rx="6" fill="#3B82F6" fill-opacity="0.16" stroke="#3B82F6" stroke-width="1.2"/>
<text x="340" y="130" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">'a'</text>
<rect x="300" y="154" width="80" height="32" rx="6" fill="#3B82F6" fill-opacity="0.16" stroke="#3B82F6" stroke-width="1.2"/>
<text x="340" y="175" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">'b'</text>
<rect x="300" y="199" width="80" height="32" rx="6" fill="#3B82F6" fill-opacity="0.16" stroke="#3B82F6" stroke-width="1.2"/>
<text x="340" y="220" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">'c'</text>
<!-- keys -> match -->
<line x1="380" y1="125" x2="464" y2="125" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.25"/>
<line x1="380" y1="170" x2="464" y2="170" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.6"/>
<line x1="380" y1="215" x2="464" y2="215" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.25"/>
<!-- match circles -->
<circle cx="480" cy="125" r="14" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-opacity="0.3" stroke-width="1.5"/>
<text x="480" y="130" font-size="13" text-anchor="middle" font-weight="500" fill="currentColor" fill-opacity="0.45">0</text>
<circle cx="480" cy="170" r="14" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.8"/>
<text x="480" y="175" font-size="13" text-anchor="middle" font-weight="600" fill="currentColor">1</text>
<circle cx="480" cy="215" r="14" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-opacity="0.3" stroke-width="1.5"/>
<text x="480" y="220" font-size="13" text-anchor="middle" font-weight="500" fill="currentColor" fill-opacity="0.45">0</text>
<!-- match -> values -->
<line x1="494" y1="125" x2="593" y2="125" stroke="#10B981" stroke-width="1.5" stroke-opacity="0.25"/>
<line x1="494" y1="170" x2="593" y2="170" stroke="#10B981" stroke-width="2.2"/>
<polygon points="600,170 593,166.5 593,173.5" fill="#10B981"/>
<line x1="494" y1="215" x2="593" y2="215" stroke="#10B981" stroke-width="1.5" stroke-opacity="0.25"/>
<!-- values -->
<rect x="600" y="109" width="80" height="32" rx="6" fill="#10B981" fill-opacity="0.05" stroke="#10B981" stroke-width="1.2" stroke-opacity="0.3"/>
<text x="640" y="130" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor" fill-opacity="0.4">1</text>
<rect x="600" y="154" width="80" height="32" rx="6" fill="#10B981" fill-opacity="0.16" stroke="#10B981" stroke-width="1.5"/>
<text x="640" y="175" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">2</text>
<rect x="600" y="199" width="80" height="32" rx="6" fill="#10B981" fill-opacity="0.05" stroke="#10B981" stroke-width="1.2" stroke-opacity="0.3"/>
<text x="640" y="220" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor" fill-opacity="0.4">3</text>
<!-- values -> result -->
<line x1="680" y1="170" x2="783" y2="170" stroke="#10B981" stroke-width="2.2"/>
<polygon points="790,170 783,166.5 783,173.5" fill="#10B981"/>
<rect x="790" y="154" width="100" height="32" rx="6" fill="#10B981" fill-opacity="0.16" stroke="#10B981" stroke-width="1.5"/>
<text x="840" y="175" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">2</text>
<!-- panel 1 notes -->
<text x="1080" y="125" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">Exactly one key wins;</text>
<text x="1080" y="141" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">every other entry</text>
<text x="1080" y="157" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">contributes nothing</text>
<text x="1080" y="185" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">No match → KeyError</text>
<!-- ============ PANEL 2: soft dictionary ============ -->
<rect x="50" y="290" width="1010" height="250" rx="14" fill="currentColor" fill-opacity="0.025" stroke="currentColor" stroke-opacity="0.35" stroke-width="1.5"/>
<text x="70" y="314" font-size="14" text-anchor="start" font-weight="600" fill="currentColor">Attention: "fuzzy" match</text>
<text x="110" y="358" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">query</text>
<text x="305" y="358" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">keys</text>
<text x="480" y="358" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">weight = softmax(q · k)</text>
<text x="655" y="358" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">values</text>
<text x="890" y="358" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">result</text>
<!-- query vector -->
<rect x="75" y="429" width="14" height="22" fill="#E8833A" fill-opacity="0.25" stroke="#E8833A" stroke-width="1"/>
<rect x="89" y="429" width="14" height="22" fill="#E8833A" fill-opacity="0.55" stroke="#E8833A" stroke-width="1"/>
<rect x="103" y="429" width="14" height="22" fill="#E8833A" fill-opacity="0.35" stroke="#E8833A" stroke-width="1"/>
<rect x="117" y="429" width="14" height="22" fill="#E8833A" fill-opacity="0.7" stroke="#E8833A" stroke-width="1"/>
<rect x="131" y="429" width="14" height="22" fill="#E8833A" fill-opacity="0.3" stroke="#E8833A" stroke-width="1"/>
<!-- query -> keys (opacity tracks weight) -->
<path d="M150,440 H215 V385 H262" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-opacity="0.35" stroke-linejoin="round"/>
<polygon points="269,385 262,381.5 262,388.5" fill="#E8833A" fill-opacity="0.35"/>
<path d="M150,440 H215 V495 H262" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-opacity="0.5" stroke-linejoin="round"/>
<polygon points="269,495 262,491.5 262,498.5" fill="#E8833A" fill-opacity="0.5"/>
<path d="M150,440 H262" fill="none" stroke="#E8833A" stroke-width="2.2" stroke-linejoin="round"/>
<polygon points="269,440 262,436.5 262,443.5" fill="#E8833A"/>
<!-- keys k1, k2, k3 -->
<rect x="270" y="374" width="14" height="22" fill="#3B82F6" fill-opacity="0.45" stroke="#3B82F6" stroke-width="1"/>
<rect x="284" y="374" width="14" height="22" fill="#3B82F6" fill-opacity="0.2" stroke="#3B82F6" stroke-width="1"/>
<rect x="298" y="374" width="14" height="22" fill="#3B82F6" fill-opacity="0.3" stroke="#3B82F6" stroke-width="1"/>
<rect x="312" y="374" width="14" height="22" fill="#3B82F6" fill-opacity="0.5" stroke="#3B82F6" stroke-width="1"/>
<rect x="326" y="374" width="14" height="22" fill="#3B82F6" fill-opacity="0.2" stroke="#3B82F6" stroke-width="1"/>
<text x="350" y="390" font-size="12" text-anchor="start" fill="currentColor">k₁</text>
<rect x="270" y="429" width="14" height="22" fill="#3B82F6" fill-opacity="0.25" stroke="#3B82F6" stroke-width="1"/>
<rect x="284" y="429" width="14" height="22" fill="#3B82F6" fill-opacity="0.55" stroke="#3B82F6" stroke-width="1"/>
<rect x="298" y="429" width="14" height="22" fill="#3B82F6" fill-opacity="0.35" stroke="#3B82F6" stroke-width="1"/>
<rect x="312" y="429" width="14" height="22" fill="#3B82F6" fill-opacity="0.65" stroke="#3B82F6" stroke-width="1"/>
<rect x="326" y="429" width="14" height="22" fill="#3B82F6" fill-opacity="0.3" stroke="#3B82F6" stroke-width="1"/>
<text x="350" y="445" font-size="12" text-anchor="start" fill="currentColor">k₂</text>
<rect x="270" y="484" width="14" height="22" fill="#3B82F6" fill-opacity="0.6" stroke="#3B82F6" stroke-width="1"/>
<rect x="284" y="484" width="14" height="22" fill="#3B82F6" fill-opacity="0.25" stroke="#3B82F6" stroke-width="1"/>
<rect x="298" y="484" width="14" height="22" fill="#3B82F6" fill-opacity="0.5" stroke="#3B82F6" stroke-width="1"/>
<rect x="312" y="484" width="14" height="22" fill="#3B82F6" fill-opacity="0.3" stroke="#3B82F6" stroke-width="1"/>
<rect x="326" y="484" width="14" height="22" fill="#3B82F6" fill-opacity="0.4" stroke="#3B82F6" stroke-width="1"/>
<text x="350" y="500" font-size="12" text-anchor="start" fill="currentColor">k₃</text>
<!-- keys -> weights -->
<line x1="378" y1="385" x2="420" y2="385" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.35"/>
<line x1="378" y1="440" x2="420" y2="440" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.6"/>
<line x1="378" y1="495" x2="420" y2="495" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.35"/>
<!-- weight bars (0.10, 0.70, 0.20) -->
<rect x="430" y="379" width="14" height="12" rx="2" fill="#E8833A" fill-opacity="0.9"/>
<text x="452" y="390" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.8">0.10</text>
<rect x="430" y="434" width="98" height="12" rx="2" fill="#E8833A" fill-opacity="0.9"/>
<text x="536" y="445" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.8">0.70</text>
<rect x="430" y="489" width="28" height="12" rx="2" fill="#E8833A" fill-opacity="0.9"/>
<text x="466" y="500" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.8">0.20</text>
<!-- weights -> values -->
<line x1="580" y1="385" x2="610" y2="385" stroke="#10B981" stroke-width="1.5" stroke-opacity="0.35"/>
<line x1="580" y1="440" x2="610" y2="440" stroke="#10B981" stroke-width="2.2"/>
<polygon points="617,440 610,436.5 610,443.5" fill="#10B981"/>
<line x1="580" y1="495" x2="610" y2="495" stroke="#10B981" stroke-width="1.5" stroke-opacity="0.45"/>
<!-- value vectors (opacity scaled by weight) -->
<rect x="620" y="374" width="14" height="22" fill="#10B981" fill-opacity="0.12" stroke="#10B981" stroke-width="1" stroke-opacity="0.4"/>
<rect x="634" y="374" width="14" height="22" fill="#10B981" fill-opacity="0.2" stroke="#10B981" stroke-width="1" stroke-opacity="0.4"/>
<rect x="648" y="374" width="14" height="22" fill="#10B981" fill-opacity="0.1" stroke="#10B981" stroke-width="1" stroke-opacity="0.4"/>
<rect x="662" y="374" width="14" height="22" fill="#10B981" fill-opacity="0.18" stroke="#10B981" stroke-width="1" stroke-opacity="0.4"/>
<rect x="676" y="374" width="14" height="22" fill="#10B981" fill-opacity="0.12" stroke="#10B981" stroke-width="1" stroke-opacity="0.4"/>
<text x="700" y="390" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.6">v₁</text>
<rect x="620" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.35" stroke="#10B981" stroke-width="1"/>
<rect x="634" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.7" stroke="#10B981" stroke-width="1"/>
<rect x="648" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.5" stroke="#10B981" stroke-width="1"/>
<rect x="662" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.3" stroke="#10B981" stroke-width="1"/>
<rect x="676" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.6" stroke="#10B981" stroke-width="1"/>
<text x="700" y="445" font-size="12" text-anchor="start" fill="currentColor">v₂</text>
<rect x="620" y="484" width="14" height="22" fill="#10B981" fill-opacity="0.2" stroke="#10B981" stroke-width="1" stroke-opacity="0.55"/>
<rect x="634" y="484" width="14" height="22" fill="#10B981" fill-opacity="0.3" stroke="#10B981" stroke-width="1" stroke-opacity="0.55"/>
<rect x="648" y="484" width="14" height="22" fill="#10B981" fill-opacity="0.45" stroke="#10B981" stroke-width="1" stroke-opacity="0.55"/>
<rect x="662" y="484" width="14" height="22" fill="#10B981" fill-opacity="0.25" stroke="#10B981" stroke-width="1" stroke-opacity="0.55"/>
<rect x="676" y="484" width="14" height="22" fill="#10B981" fill-opacity="0.35" stroke="#10B981" stroke-width="1" stroke-opacity="0.55"/>
<text x="700" y="500" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.8">v₃</text>
<!-- values -> sum -->
<path d="M726,385 H800 V432" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="800,433 796.5,426 803.5,426" fill="#10B981"/>
<line x1="726" y1="440" x2="791" y2="440" stroke="#10B981" stroke-width="2"/>
<path d="M726,495 H800 V448" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="800,447 796.5,454 803.5,454" fill="#10B981"/>
<circle cx="800" cy="440" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="795" y1="440" x2="805" y2="440" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="800" y1="435" x2="800" y2="445" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<!-- sum -> result -->
<line x1="808" y1="440" x2="850" y2="440" stroke="#10B981" stroke-width="2.2"/>
<polygon points="857,440 850,436.5 850,443.5" fill="#10B981"/>
<rect x="860" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.3" stroke="#10B981" stroke-width="1"/>
<rect x="874" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.6" stroke="#10B981" stroke-width="1"/>
<rect x="888" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.45" stroke="#10B981" stroke-width="1"/>
<rect x="902" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.3" stroke="#10B981" stroke-width="1"/>
<rect x="916" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.55" stroke="#10B981" stroke-width="1"/>
<text x="895" y="472" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">weighted blend of values</text>
<!-- panel 2 notes -->
<text x="1080" y="385" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">Every key matches a little</text>
<text x="1080" y="408" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">Weights sum to 1</text>
<text x="1080" y="431" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">Result is a blend, dominated</text>
<text x="1080" y="447" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">by the best-matching key</text>
<text x="1080" y="478" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">Smooth everywhere, so</text>
<text x="1080" y="494" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">gradients can train it</text>
<!-- ============ LEGEND ============ -->
<rect x="60" y="570" width="34" height="14" rx="3" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.2"/>
<text x="104" y="581" font-size="12" text-anchor="start" fill="currentColor">Query: what we're looking up</text>
<rect x="60" y="592" width="34" height="14" rx="3" fill="#3B82F6" fill-opacity="0.16" stroke="#3B82F6" stroke-width="1.2"/>
<text x="104" y="603" font-size="12" text-anchor="start" fill="currentColor">Keys: the labels we match against</text>
<rect x="60" y="614" width="34" height="14" rx="3" fill="#10B981" fill-opacity="0.16" stroke="#10B981" stroke-width="1.2"/>
<text x="104" y="625" font-size="12" text-anchor="start" fill="currentColor">Values: the content we get back (shading shows how much of each is used)</text>
</svg>

In [ ]:
def softmax_attention(mappings, query, temperature=1.0):
    keys = np.array([k for k, v in mappings])
    values = np.array([v for k, v in mappings])
    scores = (keys @ query) / temperature
    scores = np.exp(scores) / np.sum(np.exp(scores))
    retrieval = scores @ values
    return scores, retrieval

scores, retrieval = softmax_attention(mappings, vectors[4], temperature=0.3)

print("Retrieval:", retrieval)
plt.bar(range(len(scores)), scores)
plt.show()
for i, value in enumerate(v for k, v in mappings):
    presence = retrieval @ value
    print(f"Presence of value {i} in retrieval: {presence:.2f}")

**Something to go back and ask: how does noise change as we change: dimensionality, number of mappings, softmax temperature?**

We've now covered everything needed to understand the attention equation in the [Attention is All You Need](https://arxiv.org/pdf/1706.03762) paper.

$$\text{Attention(Q,K,V)} = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Notes:
- This is written for a whole matrix $Q$ of queries rather than a single $q$ query, and this is why the keys are transposed $K^T$.
- The temperature used is the square root of number of key-embedding dimensions $\sqrt{d_k}$. The keys and values can have different dimensions, but we've used the same number of dimensions everywhere so far.

### Fuzzy Matching

We talked a lot about trying to keep things *separate* to reduce noise. But attention gives us the ability to allow crosstalk between vectors **when being close is actually meaningful.**

For instance, suppose we define a mapping from personalities to favorite colors.

| personality type | favorite color |
| ---------------- | -------------- |
| type 1           | <span style="color: #D94A4A;font-weight: bold;">Red</span> |
| type 2           | <span style="color: #4A8FE7;font-weight: bold;">Blue</span> |
| type 3           | <span style="color: #299E69;font-weight: bold;">Green</span> |
| type 4           | <span style="color: #B58B00;font-weight: bold;">Yellow</span> |
| type 5           | <span style="color: #888888;font-weight: bold;">Gray</span> |

If we represent these as *semantically meaningful* vectors (say, an embedding for personality type, and the RGB channels for the colors), then we can use attention to predict your favorite color by your personality type.

$\text{attention}(\text{query} = \text{type 3}) \Longrightarrow$ <span style="color: #299E69; font-weight: bold;">Green</span>

So it works like a regular dictionary. But what if you're **close to a type without matching exactly?**.  
In a regular Python dictionary, there really isn't a way to represent keys being "close" to each other. But attention elegantly handles this by **interpolating** across the KV mappings.  
This works b/c the nature of attention (dot products, weighted sum) is continuous. So a small error on a personality type results in a small error on the resulting color.

$\text{attention}(\text{query} = \text{type 3} + \text{err}) \Longrightarrow$ <span style="color: #00cb1b; font-weight: bold;">Green-ish</span>

Now consider: what if the query is exactly between two personality types?  
Because of the linear nature of attention, the result is **exactly between two existing values**. This is a clear example of how attention **interpolates** across the KV space.  
(Note: this interpolation is not generally linear with softmax attention, but it's still smooth/continuous)

$\text{attention}(\text{query} = \text{mean}(\text{type 3},\ \text{type 2})) \Longrightarrow$ $\text{mean}($ <span style="color: #299E69; font-weight:bold;">Green</span> $,\ $ <span style="color: #4A8FE7; font-weight:bold;">Blue</span> $) = $ <span style="color: #0ab6a2; font-weight:bold;">Green-Blue</span>

### Backpropagation

Fuzzy matching is a result of attention being continuous. Another result of it being continuous: it's *differentiable*.

This is important for ML, because it's the *derivative of the weights with respect to the loss* which determines how to train a model. The derivative answers "how much should this weight change to decrease the model's current loss for some input-output sample?"

We can glance over the derivation for now. But just know that we are given this definition of softmax attention:

$$\text{attention}(q,K,V) = \text{softmax}\left(\frac{Kq}{\text{temperature}}\right)V$$

And we can take partial derivatives with respect to each input to determine the update rules:

$$q \leftarrow q - \eta \frac{\partial \mathcal{L}}{\partial q}$$

$$K \leftarrow K - \eta \frac{\partial \mathcal{L}}{\partial K}$$

$$V \leftarrow V - \eta \frac{\partial \mathcal{L}}{\partial V}$$

This gives us a way to train a pool of KV pairs to learn an arbitrary mapping. Let's use MSE for the loss.

Given a query $q$, target output $y$, and temperature $t$, first compute the attention weights, prediction, and error:

$$a = \operatorname{softmax}(Kq/t)$$

$$\hat{y} = V^\top a$$

$$e = \hat{y} - y$$

$$\mathcal{L} = \frac{1}{2}\|e\|^2$$

Then compute the updates:

$$g = \frac{1}{t}a \odot (Ve - \hat{y}^\top e)$$

$$K \leftarrow K - \eta gq^\top$$

$$V \leftarrow V - \eta ae^\top$$

Here, $\eta$ is the learning rate and $\odot$ is element-wise multiplication.

Let's use this to learn a continuous representation of a regular python dictionary!

We'll use a mapping of 2D positions to 3D RGB colors.  
This is also an example of the key dimensions (2D) differing from the value dimensions (3D).

In [ ]:
# Here are the points the attention will learn. Feel free to modify or add points to experiment.
# RGB sit at radius ~1 from the center; the alternates (commented out) sit at other radii,
# to show how magnitude -- not just direction -- affects how confidently a key "wins" a region.

mappings_to_learn = {
    (0.0, 1.0): (1.0, 0.0, 0.0),      # red, radius 1.0
    (-0.87, -0.5): (0.0, 1.0, 0.0),   # green, radius 1.0
    (0.87, -0.5): (0.0, 0.0, 1.0),    # blue, radius 1.0
    # (-0.35, 0.2): (1.0, 1.0, 0.0),    # yellow, radius 0.4 -- closer to center
    # (0.0, -0.6): (0.0, 1.0, 1.0),     # cyan, radius 0.6
    # (1.13, 0.65): (1.0, 0.0, 1.0),    # magenta, radius 1.3 -- farther out
}

key_coords = np.array(list(mappings_to_learn.keys()))
value_colors = np.array(list(mappings_to_learn.values()))
PLOT_RANGE = 1.6  # plot range in both x and y, centered on the origin

for (x, y), (r, g, b) in mappings_to_learn.items():
    plt.scatter(x, y, color=(r, g, b))

plt.axhline(0, color='gray', linewidth=0.8)
plt.axvline(0, color='gray', linewidth=0.8)
plt.xlim(-PLOT_RANGE, PLOT_RANGE)
plt.ylim(-PLOT_RANGE, PLOT_RANGE)
plt.gca().set_aspect('equal')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Colors at discrete points')
plt.show()

In [ ]:
def softmax_rows(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def attention_grid(key_matrix, value_matrix, queries, temperature):
    # queries: (..., d_k) -> attention output (..., d_v)
    scores = queries @ key_matrix.T / temperature
    weights = softmax_rows(scores)
    return weights @ value_matrix

training_queries = key_coords
training_targets = value_colors

grid_size = 100
x = np.linspace(-PLOT_RANGE, PLOT_RANGE, grid_size)
y = np.linspace(-PLOT_RANGE, PLOT_RANGE, grid_size)
xx, yy = np.meshgrid(x, y)
grid_queries = np.stack([xx, yy], axis=-1)

# Instead of a distance-based preview, let's plug the raw training data directly into the
# *real* attention formula -- K = positions, V = colors, no training at all -- and sweep
# temperature. Notice how we have angular wedges, not blobs: the score K_i . q / t is linear
# in q, not a function of distance to K_i, so whichever key "wins" depends on the angle
# from the centroid, not the distance -- and gets *more* confident the farther out you go
# (the score is unbounded), the opposite of how a Euclidean inverse-distance metric behaves.
temperatures_to_compare = [3, 1, 0.3, 0.1, 0.03]

fig, axes = plt.subplots(1, len(temperatures_to_compare), figsize=(4 * len(temperatures_to_compare), 4))
for ax, t in zip(axes, temperatures_to_compare):
    colors = attention_grid(training_queries, value_colors, grid_queries, t)
    ax.imshow(colors, origin='lower', extent=(-PLOT_RANGE, PLOT_RANGE, -PLOT_RANGE, PLOT_RANGE))
    ax.axhline(0, color='white', linewidth=0.5, alpha=0.5)
    ax.axvline(0, color='white', linewidth=0.5, alpha=0.5)
    ax.set_title(f'temperature = {t}')
plt.suptitle('Untrained attention: K = data positions, V = data colors')
plt.tight_layout()
plt.show()

In [ ]:
# Now let's actually learn this mapping, trained via the exact gradient update rules derived above:
#   a = softmax(Kq/t), y_hat = V^T a, e = y_hat - y
#   g = (1/t) a ⊙ (Ve - y_hat^T e)
#   K <- K - lr * g q^T, V <- V - lr * a e^T
# K and V start random; the only training signal is attention's own prediction error.

# Without NORMALIZATION_TERM, K and V have exactly enough freedom to fit every training
# point exactly, with zero pressure on what happens elsewhere -- that's the overfitting
# (wild, out-of-[0,1] extrapolation) you'll see. Turning it on adds an L2 penalty that
# pulls K, V towards smaller norms, which keeps softmax less sharp -- trading away some
# exactness at the training points to make the result look more like the untrained
# K=positions/V=colors baseline above, instead of a confident, wild extrapolation.
NORMALIZATION_TERM = False
NORMALIZATION_STRENGTH = 0.01

num_mappings = len(mappings_to_learn)
key_matrix = np.random.randn(num_mappings, 2)
value_matrix = np.random.randn(num_mappings, 3)

temperature = 0.3
learning_rate = 0.1
num_steps = 4000

for step in range(num_steps + 1):
    if step % 400 == 0:
        colors = attention_grid(key_matrix, value_matrix, grid_queries, temperature)
        plt.imshow(colors, origin='lower', extent=(-PLOT_RANGE, PLOT_RANGE, -PLOT_RANGE, PLOT_RANGE))
        plt.axhline(0, color='white', linewidth=0.5, alpha=0.5)
        plt.axvline(0, color='white', linewidth=0.5, alpha=0.5)
        plt.xlabel('X')
        plt.ylabel('Y')
        plt.title(f'Learned Mappings (Step {step})')
        plt.show()

    # one full-batch step of gradient descent on L = 1/2 sum ||attention(q) - target||^2,
    # averaged over all training points
    scores = training_queries @ key_matrix.T / temperature   # (n, n)
    a = softmax_rows(scores)                                 # (n, n)
    y_hat = a @ value_matrix                                  # (n, 3)
    e = y_hat - training_targets                              # (n, 3)
    ve = e @ value_matrix.T                                   # (n, n), row i is (V e_i)
    y_hat_dot_e = np.sum(y_hat * e, axis=1)                    # (n,)
    g = (a * (ve - y_hat_dot_e[:, None])) / temperature        # (n, n)

    d_key = g.T @ training_queries / len(training_queries)
    d_value = a.T @ e / len(training_queries)

    if NORMALIZATION_TERM:
        d_key += NORMALIZATION_STRENGTH * key_matrix
        d_value += NORMALIZATION_STRENGTH * value_matrix

    key_matrix -= learning_rate * d_key
    value_matrix -= learning_rate * d_value